In [26]:
import tensorflow as tf                          # AI library
from tensorflow import keras                     # Easy API
from tensorflow.keras import layers              # Layers
from tensorflow.keras.applications import ResNet50   # Pretrained model
from tensorflow.keras.applications.resnet50 import preprocess_input  # Image preprocess
import pathlib                                  # File path

In [27]:
data_dir = pathlib.Path("PetImages")             # Dataset folder

In [28]:
import os                      # Helps work with files and folders
import tensorflow as tf        # Used to read and check images

data_dir = "PetImages"         # Main dataset folder (Cat & Dog)

# Remove corrupted images
for folder in ["Cat", "Dog"]:   # Go inside Cat and Dog folders
    folder_path = os.path.join(data_dir, folder)   # Create full folder path
    
    for filename in os.listdir(folder_path):       # Loop through all images
        file_path = os.path.join(folder_path, filename)  # Full image path
        
        try:
            img = tf.io.read_file(file_path)       # Read image file
            tf.image.decode_jpeg(img)              # Try to open image
        except:
            os.remove(file_path)                   # Delete if image is broken

print("Corrupted images removed ")  # Done cleaning

Corrupted images removed ✅


In [29]:
IMAGE_SIZE = (224, 224)   # Each image is resized to 224x224 pixels (model needs this size)
BATCH_SIZE = 16           # Model will take 16 images together at one time (based on your RAM)
#Because ResNet50 only accepts this size 
print("Image Size:", IMAGE_SIZE) 
print("Batch Size:", BATCH_SIZE)

Image Size: (224, 224)
Batch Size: 16


In [30]:
# loads images, splits them into training & validation, and prepares them for the model
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,                    # Load images
    validation_split=0.2,        # 20% validation
    subset="training",           # Training data
    seed=123,                    # Fix random
    image_size=IMAGE_SIZE,       # Resize
    batch_size=BATCH_SIZE        # Batch size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",         # Validation data
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

print("Classes:", train_ds.class_names); 

Found 24787 files belonging to 2 classes.
Using 19830 files for training.
Found 24787 files belonging to 2 classes.
Using 4957 files for validation.
Classes: ['Cat', 'Dog']


In [31]:
# PERFORMANCE
AUTOTUNE = tf.data.AUTOTUNE              # TensorFlow automatically chooses best speed settings
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)  # Loads next data while training (faster)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)      # Same for validation (faster)

In [32]:
# DATA AUGMENTATION
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),   # Flip image left ↔ right
    layers.RandomRotation(0.1),        # Slightly rotate image
])

In [33]:
# LOAD RESNET50
base_model = ResNet50(
    input_shape=(224, 224, 3),   # Image size (224x224 with 3 colors: RGB)
    include_top=False,           # Remove original output layer
    weights='imagenet'           # Use pretrained weights (already trained)
)

base_model.trainable = False     # Freeze model (do not train it again) 

In [34]:
# BUILD MODEL (VERY SIMPLE)

inputs = keras.Input(shape=(224, 224, 3))   # Take image (224x224, RGB)

x = data_augmentation(inputs)               # Make image variations (flip/rotate)
x = preprocess_input(x)                     # Convert image format for ResNet

x = base_model(x, training=False)           # Use ResNet50 to find features (like edges, shapes)

x = layers.GlobalAveragePooling2D()(x)      # Convert big data → small summary
x = layers.BatchNormalization()(x)          # Make data stable

x = layers.Dense(128, activation='relu')(x) # Learn important patterns
x = layers.Dropout(0.5)(x)                  # Avoid overfitting (reduce memorizing)

outputs = layers.Dense(1, activation='sigmoid')(x) # Final answer (Cat or Dog)

model = keras.Model(inputs, outputs)        # Create final model

In [35]:
# COMPILE MODEL (MAKE MODEL READY)
model.compile(
    optimizer='adam',           # Helps model learn (updates weights)
    loss='binary_crossentropy', # Measures error (Cat vs Dog problem)
    metrics=['accuracy']        # Shows how correct model is
)

model.summary()                # Display model layers
print("Model ready")        # Confirmation

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)    │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ sequential_1 (Sequential)     │ (None, 224, 224, 3)       │               0 │ input_layer_4[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ get_item_3 (GetItem)          │ (None, 224, 224)          │               0 │ sequential_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ get_item_4 (GetItem)          │ (None, 224, 224)          │               0 │ sequential_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ get_item_5 (GetItem)          │ (None, 224, 224)          │               0 │ sequential_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stack_1 (Stack)               │ (None, 224, 224, 3)       │               0 │ get_item_3[0][0],          │
│                               │                           │                 │ get_item_4[0][0],          │
│                               │                           │                 │ get_item_5[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_1 (Add)                   │ (None, 224, 224, 3)       │               0 │ stack_1[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ resnet50 (Functional)         │ (None, 7, 7, 2048)        │      23,587,712 │ add_1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling2d_1    │ (None, 2048)              │               0 │ resnet50[0][0]             │
│ (GlobalAveragePooling2D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 2048)              │           8,192 │ global_average_pooling2d_… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 128)               │         262,272 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 128)               │               0 │ dense_2[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_3 (Dense)               │ (None, 1)                 │             129 │ dropout_1[0][0]            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 23,858,305 (91.01 MB)

 Trainable params: 266,497 (1.02 MB)

 Non-trainable params: 23,591,808 (90.00 MB)

Model ready ✅


In [36]:
# TRAIN (PHASE 1)
history = model.fit(
    train_ds,                 # Train data
    validation_data=val_ds,   # Validate
    epochs=3                  # Epochs
)

print("Training completed "); 

Epoch 1/3
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 1462s 1s/step - accuracy: 0.9652 - loss: 0.1022 - val_accuracy: 0.9873 - val_loss: 0.0344
Epoch 2/3
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 1450s 1s/step - accuracy: 0.9747 - loss: 0.0701 - val_accuracy: 0.9857 - val_loss: 0.0356
Epoch 3/3
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 1273s 1s/step - accuracy: 0.9779 - loss: 0.0580 - val_accuracy: 0.9865 - val_loss: 0.0382
Training completed ✅


In [37]:
# FINE-TUNING
# Fine-tuning = training only the last part of the model to improve results
base_model.trainable = True              # Allow model to learn

for layer in base_model.layers[:-30]:
    layer.trainable = False              # Keep first layers fixed (no change)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5), # Learn slowly
    loss='binary_crossentropy',         # Check error (Cat vs Dog)
    metrics=['accuracy']                # Show correct % 
)

In [38]:
# TRAIN (PHASE 2)
fine_tune_history = model.fit(
    train_ds,                 # Train data again (with fine-tuning)
    validation_data=val_ds,   # Check performance
    epochs=3                  # Number of rounds
)

# EVALUATE
model.evaluate(val_ds)       # Test model accuracy on validation data

print("Fine-tuning done")    # Done message
print("Train model again to improve accuracy and check final performance"); 

Epoch 1/3
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 1899s 2s/step - accuracy: 0.9729 - loss: 0.0717 - val_accuracy: 0.9887 - val_loss: 0.0369
Epoch 2/3
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 1888s 2s/step - accuracy: 0.9802 - loss: 0.0567 - val_accuracy: 0.9899 - val_loss: 0.0340
Epoch 3/3
1240/1240 ━━━━━━━━━━━━━━━━━━━━ 2004s 2s/step - accuracy: 0.9845 - loss: 0.0419 - val_accuracy: 0.9915 - val_loss: 0.0320
310/310 ━━━━━━━━━━━━━━━━━━━━ 324s 1s/step - accuracy: 0.9915 - loss: 0.0320
Fine-tuning done


In [58]:
model.save("cat_dog_model.keras")   # New safe format
print("Model saved in new format "); 

Model saved in new format 
